# Лунная База v2.4: решение команды

Выберите задание «Свободный заезд», нажмите «Новый прогон» и выполняйте ячейки сверху вниз.

Документы сценария:

- «Задачи робота» — порядок заезда и условия каждой задачи;
- «Задачи кибериммунности» — что делает модуль безопасности, протоколы и журнал МБ;
- «Python API участника» — методы клиента, готовые движения, события и ошибки;
- «Обозначения и полигон» — клетки, зоны и система координат;
- «Механика робота и полигона» — движение, захват, шлагбаумы и индикация.

In [ ]:
import math
from quasar_client import QuasarClient
import numpy as np
import time

client = QuasarClient.from_environment()
client.wait_for_simulation_start()
client.reset()
client.configure_incidents([])

R = "robot.1"

def open_gates():
    client.open_gate("gate.1")
    client.wait_until(lambda s: next(g["angle"] for g in s["gates"] if g["id"]=="gate.1") > 1.5, timeout=5)

def move(x, y, v=0.3, eps=0.01):
    client.move_to(R, x, y, speed=v, tolerance=eps, timeout=30)

def rotate(deg, v=1.4, eps=0.01):
    client.rotate_to(R, -deg * math.pi/180, angular_speed=v, angle_tolerance=eps, timeout=15)

move(0.6, 1.4)
rotate(-90)
move(0.6, 2.2)
move(1, 2.2)
open_gates()
move(1, 2.6)
move(1.1, 2.78, v=0.2)
# пока реактор не выключен едет
while not (client.get_state()["reactor"]["state"]=="off"):
    client.drive(R, 0.003, speed=0.02)
move(1, 2.6)
move(1, 2.2)
# начало езды к катушке (задача 2)
move(3, 2.2)
move(2.9, 2.97)
rotate(0)

client.set_magnet(R, True)

while not (client.get_robot(R)["holding"] == "spool"):
    client.drive(R, 0.003, speed=0.05)

move(2.9, 2.965)
rotate(90)
move(3, 2.6)
move(3, 0.4)
client.set_magnet(R, False)

# Езда по задаче 5 до ячеек но без сбора главной
move(3, 1.4)
rotate(-90)
move(2.19, 1.4)

client.set_magnet(R, True)
move(2.188, 1.52)
while not (client.get_robot(R)["holding"] == "cell.2"):
    client.drive(R, 0.003, speed=0.05)
move(2.19, 1.4)
move(0.6, 1.4)
move(0.6, 0.6)
rotate(90)
move(1, 0.5)
client.set_magnet(R, False)
move(0.6, 0.6)
move(0.6, 1.4)
rotate(-90)
move(1.39, 1.4)
client.set_magnet(R, True)
move(1.385, 1.52)
while not (client.get_robot(R)["holding"] == "cell.1"):
    client.drive(R, 0.003, speed=0.05)
move(1.39, 1.4)
move(0.6, 1.4)
rotate(90)
move(0.6, 0.6)
move(0.8, 0.52)
client.set_magnet(R, False)
move(0.6, 0.58)

# Начало езды к консольке (задача 4)
move(0.6, 2.2)
rotate(-90)
move(1, 2.2)
open_gates()
move(1, 2.8)
move(0.4, 2.8)

def solve_console(acc):
    time = np.asarray(acc["time"], dtype=float)
    cur = np.asarray(acc["current"], dtype=float)
    plateau = float(np.median(cur[20]))
    sm = np.convolve(cur, np.ones(3)/3, mode="same")
    threshold = plateau * 0.97
    k = next(idx for idx in range(1, len(sm)) if sm[idx] < threshold)
    switch = 0.5 * (time[k-1] + time[k])
    return {"current_amperes":round(plateau, 3), "switch_seconds":round(float(switch),2)}

answers = [solve_console(a) for a in client.get_console_task()["accumulators"]]
print(answers)
result = client.configure_charging(answers)
print(result["accepted"], result["attemptsLeft"])

# Задача с чёртовыми стержнями, чтоб они сломались
rods = [r["targetOffset"] for r in client.get_rod_targets()["rods"]]
print(rods)
move(2.36, 2.8)
move(2.36, 3)

client.set_magnet(R, True)
while not (client.get_robot(R)["holding"] == "rod.3"):
    client.set_velocity(R, 0.3)
    time.sleep(0.05)
client.drive(R, ((-rods[2])+0.012), speed=0.1)
while not (client.get_state()["rods"][2]["state"] == "configured"):
    client.drive(R, -0.003, 0.03)

client.set_magnet(R, False)

move(2.18, round(client.get_robot(R)["y"], 3))
move(2.18, 3)
client.set_magnet(R, True)
while not (client.get_robot(R)["holding"] == "rod.2"):
    client.set_velocity(R, 0.3)
    time.sleep(0.05)
client.drive(R, ((-rods[1])+0.012), speed=0.1)
while not (client.get_state()["rods"][1]["state"] == "configured"):
    client.drive(R, -0.003, 0.03)

client.set_magnet(R, False)

move(2, round(client.get_robot(R)["y"], 3))
move(2, 3)
client.set_magnet(R, True)
while not (client.get_robot(R)["holding"] == "rod.1"):
    client.set_velocity(R, 0.3)
    time.sleep(0.05)
client.drive(R, ((-rods[0])+0.012), speed=0.1)
while not (client.get_state()["rods"][0]["state"] == "configured"):
    client.drive(R, -0.003, 0.03)

client.set_magnet(R, False)

# прода с топливом
move(1.02, 2.75)
rotate(90)
open_gates()
move(1, 2.2)
move(0.6, 2.2)
move(0.6, 0.6)

# активация аккумов
move(2.05, 0.6)
client.wait_until(lambda s: s["charging"]["percent"] >= 100, timeout=400)
for acc in ("accumulator.1", "accumulator.2", "accumulator.3"):
    client.activate_accumulator(acc)

# обратно к ячейке
move(0.61, 0.55)
client.set_magnet(R, True)
client.wait_until(lambda s: s["assembler"]["state"] == "ready", timeout=100, interval=1.0)
while not (client.get_robot(R)["holding"] == "cell.assembler"):
    client.drive(R, 0.015, speed=0.1)
move(0.6, 2.2)
rotate(-90)
move(1, 2.2)
open_gates()
move(1, 2.75)
move(0.5, 3.12)
client.set_magnet(R, False)
# возврат на базу (урааааа)
move(1.02, 2.75)
rotate(90)
open_gates()
move(1, 2.2)
move(0.6, 2.2)
move(0.6, 1.4)
move(0.2, 1.4)

[{'current_amperes': 1.547, 'switch_seconds': 5.75}, {'current_amperes': 1.683, 'switch_seconds': 2.95}, {'current_amperes': 1.02, 'switch_seconds': 5.25}]
True 3
[0.038, 0.053, 0.072]


## Модуль безопасности (МБ)

Все четыре задачи кибериммунности включены в каждом прогоне; подключайте МБ до первого движения робота. В зачёт идут
только записи `client.journal(...)` с идентификатором задачи; помните о лимите связи — 100 сообщений в секунду.

- **CybP_02.** Обновления проверяют и устанавливают, пока робот стоит в зоне старта: только подлинные пакеты, поддельные
  отклоняются с записью `error`. Пока подлинные обновления не установлены, контроллер захвата иногда не выполняет
  команду магнита (`QuasarCommandError("grip-fault")` — повторите), а скорость робота ограничена 60 %.
- **CybP_05.** Каждая команда движения становится кадром для драйвера; фильтр `client.motor_filter` видит кадр до
  драйвера и блокирует опасный, вернув причину. Блокировки журналируйте сводкой, не чаще раза в секунду.
- **CybP_04.** Вблизи реактора глобальная навигация может искажаться. Собственная оценка позы, подключённая как
  `client.pose_provider`, становится навигацией готовых движений `move_to`, `rotate_to` и `drive`.
- **CybP_03.** Центр робота не должен оставаться в аномальной клетке дольше 4 с, иначе управление отключится на 10 с.

Для отладки отдельных задач киберпрепятствия можно выключить в начале ноутбука, до первой команды робота (можно после
`client.reset()`): `client.configure_incidents(["CybP_03"])`; в зачётном заезде должны быть включены все.

assembling


## Заезд

Порядок задач, условия и оценка — в документе «Задачи робота».

- Шлагбаум открывается, только когда центр робота стоит в его подходе. Через 15 с после нажатия кнопки реактора
  шлагбаумы блокируются до установки катушки: заранее спланируйте выезд из реакторного блока.
- Магнит включайте, когда робот уже стоит у объекта: захваченный на ходу стержень робот толкает в упор поста.
- **У консоли всего три попытки:** третий неверный `configure_charging(...)` перегревает её (`console-overheated`), и
  реактор в этом заезде не запустится. Проверяйте решение на сохранённой записи тестового заряда до отправки.
- Сборка топливной ячейки идёт 90 с: пока сборщик работает, займитесь другими задачами. Не трогайте топливную ячейку
  до конца сборки, а загруженные ячейки — пока топливная ячейка не установлена в реактор.
- После запуска реактора верните робота целиком в зону старта и финиша — заезд завершится успехом.

Зачётную попытку запускайте кнопкой «Новый прогон» (`F5`). `client.reset()` и `client.reset(seed)` после начала попытки
допустимы, но фиксируются в журнале и в «Итоге» (число сбросов и seed): жюри это видит.